<a href="https://www.kaggle.com/code/nihalabhay/chest-imagenet?scriptVersionId=344329735" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== STAGE 16, FOLD 1/5: EFFICIENTNETB0 ONLY (Custom CNN already done, session timed out last run) =====
!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import tensorflow as tf
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.metrics import AUC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix

FOLD_CSV_PATH = '/kaggle/input/datasets/nihalabhay/chestcvassignments/cv_fold_assignments.csv'
FINAL_CLASSES = ['no_finding', 'pathology']
IMG_SIZE, BATCH_SIZE = 224, 32
PHASE1_EPOCHS, PHASE1_LR, PHASE2_LR, EARLYSTOP_PAT = 10, 1e-3, 1e-5, 7
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
CURRENT_FOLD = 1

fold_df = pd.read_csv(FOLD_CSV_PATH)
print(f"Loaded cv_fold_assignments.csv: {len(fold_df):,} rows")
print(fold_df['fold'].value_counts().sort_index())
assert fold_df['fold'].nunique() == 5, "Fold file does not have 5 folds"
assert len(fold_df) == 44976, f"Row count {len(fold_df)} does not match the locked 44,976"

test_df   = fold_df[fold_df['fold'] == CURRENT_FOLD].reset_index(drop=True)
remainder = fold_df[fold_df['fold'] != CURRENT_FOLD].reset_index(drop=True)

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, unstratified fallback. {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_group_level_inner(df, label_col='label', val_frac=0.15, rs=SEED, tag=""):
    grouped = df.groupby('group_id')[label_col].agg(lambda s: s.value_counts().index[0]).reset_index()
    g_tr, g_va = safe_split(grouped, label_col, val_frac, rs, tag=tag)
    pick = lambda ids: df[df['group_id'].isin(ids['group_id'])]
    return pick(g_tr), pick(g_va)

train_parts, val_parts = [], []
for src in ['chex', 'nih', 'vinbig']:
    sub = remainder[remainder['source'] == src]
    tr_s, va_s = split_group_level_inner(sub, tag=f"fold{CURRENT_FOLD}-{src}-inner")
    train_parts.append(tr_s); val_parts.append(va_s)
train_df = pd.concat(train_parts, ignore_index=True)
val_df   = pd.concat(val_parts, ignore_index=True)

print(f"\nFold {CURRENT_FOLD}: Train {len(train_df):,} ({len(train_df)/len(fold_df)*100:.1f}%) | "
      f"Val {len(val_df):,} ({len(val_df)/len(fold_df)*100:.1f}%) | "
      f"Test {len(test_df):,} ({len(test_df)/len(fold_df)*100:.1f}%)")

for src in ['chex', 'nih', 'vinbig']:
    tr_g = set(train_df[train_df['source']==src]['group_id'])
    va_g = set(val_df[val_df['source']==src]['group_id'])
    te_g = set(test_df[test_df['source']==src]['group_id'])
    ok = tr_g.isdisjoint(te_g) and tr_g.isdisjoint(va_g) and va_g.isdisjoint(te_g)
    print(f"  {src}: train/val/test group-disjoint = {ok}")
    assert ok, f"LEAKAGE in fold {CURRENT_FOLD}, source {src}"
print(f"Fold {CURRENT_FOLD} leakage check: PASS (chex, nih, vinbig)")

cls = np.array(FINAL_CLASSES)
cw  = compute_class_weight('balanced', classes=cls, y=train_df['label'])
CLASS_WEIGHT = {i: w for i, w in enumerate(cw)}
print(f"Fold {CURRENT_FOLD} class_weight (recomputed fresh from THIS fold's train set):",
      {c: round(w,3) for c,w in zip(cls, cw)})

def make_gens(preprocess_fn):
    train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
    eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                  class_mode='categorical', classes=FINAL_CLASSES, color_mode='rgb')
    return (train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, **common),
            eval_idg.flow_from_dataframe(val_df, shuffle=False, **common),
            eval_idg.flow_from_dataframe(test_df, shuffle=False, **common))

def build_pretrained(base_class, num_classes=2, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                        Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

def finalise(arch_code, model_path, te_gen, live_auc):
    m = load_model(model_path)
    proba = m.predict(te_gen, verbose=0)[:,1]
    y = np.array(te_gen.classes); yhat = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y, proba); acc = accuracy_score(y, yhat)
    f1 = f1_score(y, yhat, average='macro')
    tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
    sens = tp/(tp+fn) if (tp+fn) else float('nan')
    spec = tn/(tn+fp) if (tn+fp) else float('nan')
    del m; import gc; gc.collect()

    print(f"Live AUC={live_auc:.4f} | Reloaded AUC={auc:.4f} | Match: {abs(live_auc-auc) < 1e-4}")
    assert abs(live_auc - auc) < 1e-4, "CHECKPOINT MISMATCH, invalid provenance, discard this result"

    np.savez_compressed(f'/kaggle/working/cv_f{CURRENT_FOLD}_preds_{arch_code}.npz',
                        proba=proba, y_true=y,
                        source=test_df['source'].values, group_id=test_df['group_id'].values)
    return {'fold':CURRENT_FOLD, 'arch':arch_code, 'accuracy':acc, 'macro_f1':f1, 'macro_auc':auc,
            'macro_sensitivity':sens, 'macro_specificity':spec, 'n_test':len(y),
            'tn':tn, 'fp':fp, 'fn':fn, 'tp':tp}

# ---- EFFICIENTNETB0 ----
print(f"\n{'='*20} FOLD {CURRENT_FOLD}: EFFICIENTNETB0 {'='*20}")
tr, va, te = make_gens(eff_pre)
model, base = build_pretrained(EfficientNetB0)
base.trainable = False
model.compile(Adam(PHASE1_LR), 'categorical_crossentropy', ['accuracy', AUC(name='auc')])
model.fit(tr, validation_data=va, epochs=PHASE1_EPOCHS, class_weight=CLASS_WEIGHT, verbose=1,
          callbacks=[CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_eff_log.csv', append=False)])
model.layers[0].trainable = True
model.compile(Adam(PHASE2_LR), 'categorical_crossentropy', ['accuracy', AUC(name='auc')])
p = f'/kaggle/working/cv_f{CURRENT_FOLD}_eff.keras'
model.fit(tr, validation_data=va, epochs=60, class_weight=CLASS_WEIGHT, verbose=1,
          callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=EARLYSTOP_PAT, restore_best_weights=True),
                     ModelCheckpoint(p, monitor='val_auc', mode='max', save_best_only=True),
                     CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_eff_log.csv', append=True)])
live = roc_auc_score(np.array(te.classes), model.predict(te, verbose=0)[:,1])
del model, base; import gc; gc.collect()
eff_row = finalise('eff', p, te, live)
pd.DataFrame([eff_row]).to_csv(f'/kaggle/working/cv_f{CURRENT_FOLD}_eff_results.csv', index=False)

print(f"\n\n{'='*20} FOLD {CURRENT_FOLD}: EFFICIENTNETB0 DONE {'='*20}")
print(pd.DataFrame([eff_row]).to_string(index=False))
print(f"\nDownload cv_f{CURRENT_FOLD}_eff.keras, cv_f{CURRENT_FOLD}_eff_log.csv, "
      f"cv_f{CURRENT_FOLD}_preds_eff.npz, cv_f{CURRENT_FOLD}_eff_results.csv individually now.")
print(f"Fold {CURRENT_FOLD} status: custom done (earlier session), eff done (this session). "
      f"Still needed for fold {CURRENT_FOLD}: mob, res. Do not start fold 2 until all 4 exist.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 114.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-23 09:05:04.105933: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787475904.131481      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787475904.139292      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787475904.159773      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787475904.159792      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787475904.159794      24 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Loaded cv_fold_assignments.csv: 44,976 rows
fold
1    9068
2    9099
3    8915
4    8975
5    8919
Name: count, dtype: int64

Fold 1: Train 30,569 (68.0%) | Val 5,339 (11.9%) | Test 9,068 (20.2%)
  chex: train/val/test group-disjoint = True
  nih: train/val/test group-disjoint = True
  vinbig: train/val/test group-disjoint = True
Fold 1 leakage check: PASS (chex, nih, vinbig)
Fold 1 class_weight (recomputed fresh from THIS fold's train set): {np.str_('no_finding'): np.float64(1.115), np.str_('pathology'): np.float64(0.907)}

==================== FOLD 1: EFFICIENTNETB0 ====================
Found 30569 validated image filenames belonging to 2 classes.
Found 5339 validated image filenames belonging to 2 classes.
Found 9068 validated image filenames belonging to 2 classes.


I0000 00:00:1787476096.101671      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787476096.107940      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 [==============================] - 0s 0us/step
Epoch 1/10


E0000 00:00:1787476105.945474      24 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1787476107.282785      74 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1787476109.397998      76 service.cc:152] XLA service 0x7fd454eb0a80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787476109.398030      76 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1787476109.398034      76 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1787476109.550905      76 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


956/956 [==============================] - 867s 898ms/step - loss: 0.4833 - accuracy: 0.7659 - auc: 0.8476 - val_loss: 0.4469 - val_accuracy: 0.7930 - val_auc: 0.8744
Epoch 2/10
956/956 [==============================] - 647s 677ms/step - loss: 0.4483 - accuracy: 0.7848 - auc: 0.8707 - val_loss: 0.4413 - val_accuracy: 0.7883 - val_auc: 0.8757
Epoch 3/10
956/956 [==============================] - 658s 688ms/step - loss: 0.4339 - accuracy: 0.7948 - auc: 0.8801 - val_loss: 0.4285 - val_accuracy: 0.7919 - val_auc: 0.8828
Epoch 4/10
956/956 [==============================] - 663s 693ms/step - loss: 0.4220 - accuracy: 0.8016 - auc: 0.8868 - val_loss: 0.4092 - val_accuracy: 0.8056 - val_auc: 0.8939
Epoch 5/10
956/956 [==============================] - 663s 693ms/step - loss: 0.4188 - accuracy: 0.8040 - auc: 0.8889 - val_loss: 0.4341 - val_accuracy: 0.7936 - val_auc: 0.8821
Epoch 6/10
956/956 [==============================] - 661s 692ms/step - loss: 0.4153 - accuracy: 0.8035 - auc: 0.8907 - v

E0000 00:00:1787483041.858489      24 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


956/956 [==============================] - 907s 905ms/step - loss: 0.5820 - accuracy: 0.7364 - auc: 0.8131 - val_loss: 0.5150 - val_accuracy: 0.7674 - val_auc: 0.8463
Epoch 2/60
956/956 [==============================] - 850s 889ms/step - loss: 0.4759 - accuracy: 0.7777 - auc: 0.8617 - val_loss: 0.4630 - val_accuracy: 0.7857 - val_auc: 0.8680
Epoch 3/60
956/956 [==============================] - 859s 898ms/step - loss: 0.4381 - accuracy: 0.7950 - auc: 0.8810 - val_loss: 0.4598 - val_accuracy: 0.7934 - val_auc: 0.8728
Epoch 4/60
956/956 [==============================] - 847s 886ms/step - loss: 0.4164 - accuracy: 0.8057 - auc: 0.8918 - val_loss: 0.4541 - val_accuracy: 0.7985 - val_auc: 0.8794
Epoch 5/60
956/956 [==============================] - 842s 881ms/step - loss: 0.4045 - accuracy: 0.8112 - auc: 0.8980 - val_loss: 0.4277 - val_accuracy: 0.8048 - val_auc: 0.8879
Epoch 6/60
956/956 [==============================] - 837s 876ms/step - loss: 0.3943 - accuracy: 0.8166 - auc: 0.9027 - v